# Full-profile physics-only channel operator
Complete supplied spline in hard inlet boundary; variable-quadrature function encoder.
No CFD response labels in the objective. Benchmark references, if present, are read-only monitors.
Candidate run: successful execution alone does not certify prediction accuracy.


In [ ]:
import os,sys,json,base64,subprocess
from pathlib import Path
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE']='false'
os.environ['MPLCONFIGDIR']='/kaggle/working/.mpl'
os.environ['OPENBLAS_NUM_THREADS']='1'
os.environ['OMP_NUM_THREADS']='1'
import jax
print('JAX',jax.__version__,'DEVICES',jax.devices())
assert any(d.platform=='gpu' for d in jax.devices()), 'GPU required'
sources={'flow_operator/__init__.py': '"""Physics-only operator for a supplied continuous channel-inlet profile."""\n', 'flow_operator/api.py': '"""Reusable frozen-weight inference with one JIT cache for successive inlets."""\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nfrom .checkpoint import load\nfrom .model import make_functions\nfrom .profiles import stack\n\n\nclass FlowOperator:\n    def __init__(self,checkpoint,allow_unvalidated=False,chunk_size=1024):\n        self.path=Path(checkpoint)\n        self.sha256=hashlib.sha256(self.path.read_bytes()).hexdigest()\n        report_path=self.path.with_suffix(\'.validation.json\')\n        report=json.loads(report_path.read_text()) if report_path.exists() else {}\n        self.benchmark_accepted=(report.get(\'checkpoint_sha256\')==self.sha256 and report.get(\'all_gates_passed\') is True)\n        if not self.benchmark_accepted and not allow_unvalidated:\n            raise ValueError(\'No passing benchmark report for these weights; provisional use requires allow_unvalidated=True\')\n        if chunk_size<1:raise ValueError(\'chunk_size must be positive\')\n        self.params,self.config,self.metadata,_=load(checkpoint)\n        self.functions=make_functions(self.config);self.chunk_size=chunk_size\n\n    def evaluate(self,inlet,reynolds,xy):\n        """Return dimensionless [u,v,p]; no optimization, CFD, or response lookup."""\n        xy=np.asarray(xy,dtype=np.float32)\n        if xy.ndim!=2 or xy.shape[1]!=2 or not np.isfinite(xy).all():raise ValueError(\'xy must be finite with shape (N,2)\')\n        if not np.isfinite(reynolds) or reynolds<=0:raise ValueError(\'Reynolds must be positive and finite\')\n        if np.any(xy<0) or np.any(xy[:,0]>self.config.length) or np.any(xy[:,1]>1):raise ValueError(\'Requested coordinates are outside the channel\')\n        if not len(xy):return np.empty((0,3))\n        data=stack([inlet]);re=np.array([reynolds],np.float32);result=[]\n        for left in range(0,len(xy),self.chunk_size):\n            part=xy[left:left+self.chunk_size];actual=len(part)\n            padded=np.pad(part,((0,self.chunk_size-actual),(0,0)),mode=\'edge\')\n            fields=np.asarray(self.functions[\'fields\'](self.params,data,re,padded[None]))[0,:actual]\n            if not np.isfinite(fields).all():raise RuntimeError(\'Nonfinite prediction\')\n            result.append(fields)\n        return np.concatenate(result)\n', 'flow_operator/checkpoint.py': '"""Non-pickle versioned checkpoint with config, optimizer, and RNG provenance."""\nimport json\nfrom pathlib import Path\n\nimport jax\nimport jax.numpy as jnp\nimport numpy as np\nfrom .model import Config, initialize\n\nFORMAT=\'full_profile_physics_operator_v1\'\n\n\ndef save(path,params,config,step,rng_state,optimizer=None,metadata=None):\n    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)\n    flat=jax.tree_util.tree_leaves(params)\n    arrays={f\'parameter_{i}\':np.array(v) for i,v in enumerate(flat)}\n    if optimizer is not None:\n        for name,tree in zip([\'adam_m\',\'adam_v\'],optimizer):\n            arrays.update({f\'{name}_{i}\':np.array(v) for i,v in enumerate(jax.tree_util.tree_leaves(tree))})\n    arrays[\'metadata\']=np.array(json.dumps(dict(format=FORMAT,config=config.json(),step=int(step),\n        rng_state=rng_state,optimizer_present=optimizer is not None,**(metadata or {}))))\n    temporary=path.with_suffix(\'.tmp.npz\')\n    np.savez_compressed(temporary,**arrays);temporary.replace(path)\n\n\ndef load(path,optimizer=False):\n    with np.load(path,allow_pickle=False) as d:\n        metadata=json.loads(str(d[\'metadata\']))\n        if metadata[\'format\']!=FORMAT:raise ValueError(\'Not a full-profile checkpoint; old V2 weights are incompatible\')\n        # The initial Gaussian experiment predates the explicit lift field.\n        configuration=dict(metadata[\'config\']);configuration.setdefault(\'lift\',\'gaussian\')\n        config=Config(**configuration);template=initialize(config)\n        flat,structure=jax.tree_util.tree_flatten(template)\n        def tree(prefix):\n            values=[jnp.asarray(d[f\'{prefix}_{i}\']) for i in range(len(flat))]\n            if any(a.shape!=b.shape for a,b in zip(values,flat)):raise ValueError(\'Checkpoint parameter shape mismatch\')\n            return jax.tree_util.tree_unflatten(structure,values)\n        params=tree(\'parameter\');state=None\n        if optimizer:\n            if not metadata[\'optimizer_present\']:raise ValueError(\'Optimizer is missing\')\n            state=(tree(\'adam_m\'),tree(\'adam_v\'))\n    return params,config,metadata,state\n', 'flow_operator/evaluate.py': '"""Evaluate frozen weights against independent references; never train here."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport jax\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nimport numpy as np\nfrom .checkpoint import load\nfrom .model import make_functions\nfrom .profiles import stack\nfrom .validation import read_cases,compare\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--checkpoint\',required=True);p.add_argument(\'--references\',required=True)\n    p.add_argument(\'--output\',required=True);p.add_argument(\'--acceptance\',default=\'configs/acceptance.json\')\n    p.add_argument(\'--role\',choices=[\'repeated_validation\',\'final_heldout_test\'],default=\'repeated_validation\')\n    a=p.parse_args();out=Path(a.output);out.mkdir(parents=True,exist_ok=True)\n    checkpoint=Path(a.checkpoint);sha=hashlib.sha256(checkpoint.read_bytes()).hexdigest()\n    params,cfg,metadata,_=load(checkpoint)\n    # Diagnostics in double precision from the actual trained float32 parameters.\n    jax.config.update(\'jax_enable_x64\',True)\n    params=jax.tree_util.tree_map(lambda x:x.astype(np.float64),params)\n    functions=make_functions(cfg);cases=read_cases(a.references)\n    field_report=compare(params,functions,cases);rng=np.random.default_rng(918397)\n    limits=json.loads(Path(a.acceptance).read_text())\n    for case,record in zip(cases,field_report[\'cases\']):\n        data=stack([case[\'profile\']],np.float64);re=np.array([case[\'re\']])\n        xy=rng.uniform([.001,.001],[cfg.length-.001,.999],size=(1,1024,2));xy[...,:1]=cfg.length*(xy[...,:1]/cfg.length)**2\n        residual=np.asarray(functions[\'residual\'](params,data,re,xy))[0]\n        y=np.linspace(0,1,1025)\n        inlet_xy=np.column_stack([np.zeros_like(y),y])[None]\n        fields=np.asarray(functions[\'fields\'](params,data,re,inlet_xy))[0]\n        record.update(momentum_rms=float(np.sqrt(np.mean(np.sum(residual[:,:2]**2,axis=1)))),\n            divergence_rms=float(np.sqrt(np.mean(residual[:,2]**2))),\n            boundary_max_error=float(max(np.max(abs(fields[:,0]-case[\'profile\'].spline(y))),np.max(abs(fields[:,1])))))\n        boundary=np.asarray(functions[\'outlet\'](params,data,re,np.linspace(.001,.999,257)[None]))[0]\n        record[\'outlet_traction_rms\']=float(np.sqrt(np.mean(np.sum(boundary**2,axis=1))))\n        xx,yy=np.meshgrid(np.linspace(0,cfg.length,31),np.linspace(0,1,513),indexing=\'ij\')\n        grid_xy=np.stack([xx,yy],-1).reshape(-1,2)\n        parts=[np.asarray(functions[\'fields\'](params,data,re,z[None]))[0]\n               for z in np.array_split(grid_xy,int(np.ceil(len(grid_xy)/1024)))]\n        grid=np.concatenate(parts).reshape(31,513,3)\n        record[\'flux_max_error\']=float(np.max(abs(np.trapezoid(grid[...,0],yy,axis=1)-1)))\n        # Save reference + actual model outputs, without gauge fitting.\n        pred=np.concatenate([np.asarray(functions[\'fields\'](params,data,re,z[None]))[0]\n             for z in np.array_split(case[\'xy\'],int(np.ceil(len(case[\'xy\'])/1024)))])\n        np.savez_compressed(out/(case[\'name\']+\'.npz\'),xy=case[\'xy\'],reference=case[\'reference\'],prediction=pred,\n                            inlet_y=y,inlet_requested=case[\'profile\'].spline(y),inlet_applied=fields[:,0])\n        nx,ny=len(np.unique(case[\'xy\'][:,0])),len(np.unique(case[\'xy\'][:,1]))\n        fig,axes=plt.subplots(3,3,figsize=(15,9),layout=\'constrained\')\n        for row,label in enumerate([\'u\',\'v\',\'p\']):\n            reference=case[\'reference\'][:,row].reshape(nx,ny).T;model=pred[:,row].reshape(nx,ny).T\n            low,high=min(reference.min(),model.min()),max(reference.max(),model.max())\n            for col,(title,value) in enumerate([(\'FEM\',reference),(\'Frozen full-profile operator\',model),(\'model - FEM\',model-reference)]):\n                kwargs=dict(vmin=low,vmax=high) if col<2 else dict(vmin=-np.max(abs(value))-1e-15,vmax=np.max(abs(value))+1e-15)\n                im=axes[row,col].pcolormesh(np.unique(case[\'xy\'][:,0]),np.unique(case[\'xy\'][:,1]),value,\n                     shading=\'nearest\',cmap=\'viridis\' if col<2 else \'RdBu_r\',**kwargs)\n                axes[row,col].set(title=f\'{label}: {title}\',xlabel=\'x/H\',ylabel=\'y/H\')\n                fig.colorbar(im,ax=axes[row,col],format=\'%.3g\')\n        fig.suptitle(f"{case[\'name\']} | Re={case[\'re\']:g} | velocity error {100*record[\'velocity_relative_l2\']:.2f}% | pressure error {100*record[\'pressure_relative_l2\']:.2f}%")\n        fig.savefig(out/(case[\'name\']+\'.png\'),dpi=140);plt.close(fig)\n    thresholds={k:v for k,v in limits.items() if isinstance(v,(int,float))}\n    gates={k:all(r[k]<v for r in field_report[\'cases\']) for k,v in thresholds.items()}\n    report=dict(checkpoint_sha256=sha,step=metadata[\'step\'],role=a.role,cases=field_report[\'cases\'],\n        thresholds=thresholds,gates=gates,all_gates_passed=all(gates.values()),\n        universal_accuracy_guaranteed=False,training_labels_used=metadata[\'CFD_response_labels_used\'],\n        note=\'Accuracy measured on the listed benchmark only; no pressure alignment or per-input training.\')\n    (out/\'metrics.json\').write_text(json.dumps(report,indent=2))\n    # The predictor accepts only a matching, passing report next to the checkpoint.\n    checkpoint.with_suffix(\'.validation.json\').write_text(json.dumps(report,indent=2))\n    assert hashlib.sha256(checkpoint.read_bytes()).hexdigest()==sha\n    print(json.dumps(report,indent=2))\n\n\nif __name__==\'__main__\':main()\n', 'flow_operator/model.py': '"""Variable-quadrature function encoder + full-profile streamfunction DeepONet.\n\nThe input shape is not truncated to a Fourier basis. Positional features enrich\nthe neural encoder/trunk; they never replace the function in the hard boundary.\n"""\nfrom dataclasses import dataclass, asdict\n\nimport jax\nimport jax.numpy as jnp\nimport numpy as np\n\n\n@dataclass(frozen=True)\nclass Config:\n    width: int = 96\n    latent: int = 48\n    depth: int = 3\n    encoder_width: int = 48\n    length: float = 5.\n    seed: int = 91837\n    lift: str = \'gaussian\'\n    functional_moments: bool = False\n    encoder_kind: str = \'pooled\'\n    sensor_count: int = 257\n\n    def __post_init__(self):\n        if min(self.width,self.latent,self.depth,self.encoder_width)<1 or self.sensor_count<16:\n            raise ValueError(\'Network dimensions must be positive and sensor_count >=16\')\n        if self.lift not in [\'gaussian\',\'oseen\'] or self.encoder_kind not in [\'pooled\',\'sensor\']:\n            raise ValueError(\'Unknown model configuration\')\n        if self.encoder_kind==\'sensor\' and self.functional_moments:\n            raise ValueError(\'The direct sensor branch already receives the full sensor vector; disable functional_moments\')\n\n    def json(self): return asdict(self)\n\n\nFREQUENCIES = tuple(float(k) for k in range(1,49))\n\n\ndef initialize(config=Config()):\n    rng = np.random.default_rng(config.seed)\n    def layer(n,m,scale=None):\n        return dict(w=jnp.array(rng.normal(0,np.sqrt(2/(n+m)) if scale is None else scale,(n,m)),dtype=jnp.float32),\n                    b=jnp.zeros(m,dtype=jnp.float32))\n    def mlp(n,m,width,depth):\n        return dict(u=layer(n,width),v=layer(n,width),\n                    gates=[layer(n if i==0 else width,width) for i in range(depth)], out=layer(width,m,1e-3))\n    # Pointwise encoder weights do not depend on the number/locations of samples.\n    branch_inputs=config.sensor_count+2 if config.encoder_kind==\'sensor\' else 2*config.encoder_width+2+(2*len(FREQUENCIES)+1 if config.functional_moments else 0)\n    params=dict(branch=mlp(branch_inputs,2*config.latent,config.width,config.depth),\n                trunk=mlp(10+2*len(FREQUENCIES),2*config.latent,config.width,config.depth))\n    if config.encoder_kind==\'pooled\':\n        params[\'encoder\']=[layer(2+2*len(FREQUENCIES),config.encoder_width),layer(config.encoder_width,config.encoder_width)]\n    return params\n\n\ndef dense(p,x): return x@p[\'w\']+p[\'b\']\n\n\ndef mlp(p,x):\n    u,v=jnp.tanh(dense(p[\'u\'],x)),jnp.tanh(dense(p[\'v\'],x))\n    h=x\n    for gate in p[\'gates\']:\n        z=jnp.tanh(dense(gate,h));h=z*u+(1-z)*v\n    return dense(p[\'out\'],h)\n\n\ndef spline(profile,y):\n    index=jnp.clip(jnp.searchsorted(profile[\'knots\'],y,side=\'right\')-1,0,len(profile[\'knots\'])-2)\n    left=profile[\'knots\'][index];h=profile[\'knots\'][index+1]-left\n    t=(y-left)/h\n    a,b,c,d=profile[\'coeff\'][:,index]\n    f=((a*t+b)*t+c)*t+d\n    integral=profile[\'prefix\'][index]+h*(((a*t/4+b/3)*t+c/2)*t+d)*t\n    return f,integral\n\n\ndef context(params,profile,re,config):\n    if config.encoder_kind==\'sensor\':\n        # The paper\'s direct-sensor branch: ALL samples, not pooled moments.\n        # The supplied spline itself (possibly with a different knot grid) still\n        # enters the hard boundary unchanged. This grid is only the branch input.\n        sensors=jnp.linspace(0.,1.,config.sensor_count)\n        sampled=jax.vmap(lambda y:spline(profile,y)[0])(sensors)\n        inputs=jnp.concatenate([sampled-6*sensors*(1-sensors),\n            jnp.array([jnp.log(re/10)/jnp.log(10.),jnp.log1p(profile[\'effective_mode\'])/5])])\n        return mlp(params[\'branch\'],inputs).reshape(2,config.latent)\n    y,f,w=profile[\'encoder_y\'],profile[\'encoder_f\'],profile[\'encoder_w\']\n    phase=jnp.pi*y[:,None]*jnp.array(FREQUENCIES)\n    encoded=jnp.concatenate([y[:,None],(f-6*y*(1-y))[:,None],jnp.sin(phase),jnp.cos(phase)],axis=1)\n    for p in params[\'encoder\']: encoded=jnp.tanh(dense(p,encoded))\n    mean=jnp.sum(w[:,None]*encoded,axis=0)\n    second=jnp.sum(w[:,None]*encoded**2,axis=0)\n    inputs=jnp.concatenate([mean,second,jnp.array([jnp.log(re/10)/jnp.log(10.),jnp.log1p(profile[\'effective_mode\'])/5])])\n    if config.functional_moments:\n        # Strong condition features, alongside the learned pointwise encoder.\n        # They do NOT reconstruct or replace the inlet spline in the boundary.\n        delta=f-6*y*(1-y)\n        moments=jnp.concatenate([jnp.sum((w*delta)[:,None]*jnp.sin(phase),axis=0),\n                                 jnp.sum((w*delta)[:,None]*jnp.cos(phase),axis=0),\n                                 jnp.array([jnp.sum(w*delta*delta)])])\n        inputs=jnp.concatenate([inputs,moments])\n    return mlp(params[\'branch\'],inputs).reshape(2,config.latent)\n\n\ndef potentials(params,branch,profile,re,xy,config):\n    x,y=xy\n    modes=jnp.array(FREQUENCIES)\n    phase=jnp.pi*y*modes\n    # These are coordinate embeddings, NOT a projection of the input function.\n    # Amplitude scaling conditions high derivatives.\n    features=jnp.concatenate([jnp.array([2*x/config.length-1,2*y-1]),\n        jnp.sin(phase)/modes,jnp.cos(phase)/modes,\n        jnp.array([jnp.exp(-x/s) for s in [.02,.05,.1,.3,1.,3.]]),\n        jnp.array([jnp.log1p(20*x)/jnp.log1p(20*config.length),x/config.length])])\n    raw=jnp.sum(branch*mlp(params[\'trunk\'],features).reshape(2,config.latent),axis=1)/jnp.sqrt(float(config.latent))\n    inlet,integral=spline(profile,y)\n    base=3*y*y-2*y**3\n    pressure_lift=0.\n    if config.lift==\'oseen\':\n        # Linearized half-plane NS lift. This is an analytic prior, not a CFD\n        # label or a replacement for the full nonlinear NS objective.\n        k=jnp.pi*profile[\'effective_mode\']\n        alpha=2*k*k/(jnp.sqrt(re*re+4*k*k)+re)\n        delta=re*alpha/(k+alpha)\n        decay=jnp.exp(-alpha*x)*(1-alpha*jnp.expm1(-delta*x)/delta)\n        pressure_lift=(k+alpha)/re*(inlet-6*y*(1-y))*jnp.exp(-k*x)\n    elif config.lift==\'gaussian\':\n        ell=.05+.05*re/profile[\'effective_mode\']**2\n        decay=jnp.exp(-(x/ell)**2)\n    else:\n        raise ValueError(\'Unknown lift configuration\')\n    # Entire spline antiderivative, including ALL supplied oscillations.\n    lift=base+(integral-base)*decay\n    mask=16*y*y*(1-y)**2*(1-jnp.exp(-x/.1))**2\n    psi=lift+mask*raw[0]\n    pressure=12/re*(config.length-x)+pressure_lift+raw[1]\n    return psi,pressure\n\n\ndef point_fields(params,branch,profile,re,xy,config):\n    grad=jax.grad(lambda z:potentials(params,branch,profile,re,z,config)[0])(xy)\n    return jnp.array([grad[1],-grad[0],potentials(params,branch,profile,re,xy,config)[1]])\n\n\ndef make_functions(config=Config()):\n    jac=jax.jacfwd(point_fields,argnums=4)\n    hess=jax.jacfwd(jac,argnums=4)\n    def point_residual(params,branch,profile,re,xy):\n        u,v,p=point_fields(params,branch,profile,re,xy,config)\n        g=jac(params,branch,profile,re,xy,config);h=hess(params,branch,profile,re,xy,config)\n        return jnp.array([u*g[0,0]+v*g[0,1]+g[2,0]-(h[0,0,0]+h[0,1,1])/re,\n            u*g[1,0]+v*g[1,1]+g[2,1]-(h[1,0,0]+h[1,1,1])/re,g[0,0]+g[1,1]])\n    def outlet_one(params,branch,profile,re,y):\n        xy=jnp.array([config.length,y]);f=point_fields(params,branch,profile,re,xy,config)\n        g=jac(params,branch,profile,re,xy,config)\n        return jnp.array([g[0,0]/re-f[2],g[1,0]/re])\n    def one_fields(params,profile,re,xy):\n        branch=context(params,profile,re,config)\n        return jax.vmap(lambda z:point_fields(params,branch,profile,re,z,config))(xy)\n    def one_residual(params,profile,re,xy):\n        branch=context(params,profile,re,config)\n        return jax.vmap(lambda z:point_residual(params,branch,profile,re,z))(xy)\n    def one_outlet(params,profile,re,y):\n        branch=context(params,profile,re,config)\n        return jax.vmap(lambda z:outlet_one(params,branch,profile,re,z))(y)\n    fields=jax.vmap(one_fields,in_axes=(None,0,0,0))\n    residual=jax.vmap(one_residual,in_axes=(None,0,0,0))\n    outlet=jax.vmap(one_outlet,in_axes=(None,0,0,0))\n    def terms(params,batch):\n        r=residual(params,batch[\'profiles\'],batch[\'reynolds\'],batch[\'xy\'])\n        o=outlet(params,batch[\'profiles\'],batch[\'reynolds\'],batch[\'outlet_y\'])\n        return jnp.array([jnp.mean(jnp.sum(r[...,:2]**2,axis=-1)),jnp.mean(jnp.sum(o**2,axis=-1)),jnp.mean(r[...,2]**2)])\n    return dict(fields=jax.jit(fields),residual=jax.jit(residual),outlet=jax.jit(outlet),terms=jax.jit(terms),raw_terms=terms)\n', 'flow_operator/predict.py': '"""Fixed-weight evaluation; never optimize weights or call a reference solver."""\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport time\n\nimport jax\nimport numpy as np\nfrom .profiles import Inlet,stack\nfrom .checkpoint import load\nfrom .model import make_functions,context\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--checkpoint\',required=True);p.add_argument(\'--inlet-csv\',required=True)\n    p.add_argument(\'--re\',type=float,required=True);p.add_argument(\'--output\',default=\'prediction.npz\')\n    p.add_argument(\'--nx\',type=int,default=151);p.add_argument(\'--ny\',type=int,default=129)\n    p.add_argument(\'--encoder-points\',type=int,default=257);p.add_argument(\'--allow-unvalidated\',action=\'store_true\')\n    a=p.parse_args()\n    if a.re<=0 or min(a.nx,a.ny)<2:p.error(\'Require Re>0 and at least two points per axis\')\n    checkpoint=Path(a.checkpoint);sha=hashlib.sha256(checkpoint.read_bytes()).hexdigest()\n    report_path=checkpoint.with_suffix(\'.validation.json\')\n    accepted=False\n    if report_path.exists():\n        validation=json.loads(report_path.read_text())\n        accepted=validation.get(\'checkpoint_sha256\')==sha and validation.get(\'all_gates_passed\') is True\n    if not accepted and not a.allow_unvalidated:\n        p.error(\'No matching accepted validation report. Use --allow-unvalidated only for an explicitly provisional diagnostic.\')\n    if not accepted:\n        print(\'PROVISIONAL: this checkpoint has not passed the registered independent benchmark.\')\n    profile=Inlet.from_csv(a.inlet_csv,a.encoder_points)\n    params,config,metadata,_=load(checkpoint)\n    functions=make_functions(config);data=stack([profile])\n    x,y=np.linspace(0,config.length,a.nx),np.linspace(0,1,a.ny)\n    xx,yy=np.meshgrid(x,y,indexing=\'ij\');xy=np.stack([xx,yy],-1).reshape(-1,2)\n    start=time.perf_counter();blocks=[]\n    for z in np.array_split(xy,int(np.ceil(len(xy)/1024))):\n        blocks.append(np.asarray(functions[\'fields\'](params,data,np.array([a.re],np.float32),z[None].astype(np.float32)))[0])\n    result=np.concatenate(blocks).reshape(len(x),len(y),3)\n    if not np.isfinite(result).all():raise RuntimeError(\'Nonfinite prediction\')\n    elapsed=time.perf_counter()-start\n    check_y=np.linspace(0,1,1025);check_xy=np.stack([np.zeros_like(check_y),check_y],-1)\n    inlet=np.asarray(functions[\'fields\'](params,data,np.array([a.re],np.float32),check_xy[None].astype(np.float32)))[0]\n    diag=profile.diagnostics()\n    diag.update(checkpoint_sha256=sha,training_step=metadata[\'step\'],retraining=False,\n        benchmark_accepted=accepted,universal_accuracy_guaranteed=False,\n        seconds_including_first_JIT=elapsed,reynolds=a.re,\n        outside_trained_re_interval=not 10<=a.re<=100,\n        boundary_max_error=float(np.max(abs(inlet[:,0]-profile.spline(check_y)))),\n        boundary_v_max=float(np.max(abs(inlet[:,1]))),\n        inference=\'Function evaluation with frozen weights; no optimizer, CFD, or response-bank lookup\',\n        pressure_reference=\'Unsymmetric-gradient do-nothing traction, exterior pressure zero\')\n    # Independently probe the physical residual; do not modify this prediction.\n    test_xy=np.random.default_rng(91897).uniform([.001,.001],[config.length-.001,.999],size=(1,256,2)).astype(np.float32)\n    residual=np.asarray(functions[\'residual\'](params,data,np.array([a.re],np.float32),test_xy))[0]\n    diag[\'momentum_rms\']=float(np.sqrt(np.mean(np.sum(residual[:,:2]**2,axis=1))))\n    diag[\'divergence_rms\']=float(np.sqrt(np.mean(residual[:,2]**2)))\n    output=Path(a.output);output.parent.mkdir(parents=True,exist_ok=True)\n    np.savez_compressed(output,x=x,y=y,u=result[...,0],v=result[...,1],p=result[...,2],inlet_y=check_y,\n                        inlet_requested=profile.spline(check_y),inlet_applied=inlet[:,0],reynolds=a.re)\n    output.with_suffix(\'.json\').write_text(json.dumps(diag,indent=2))\n    print(json.dumps(diag,indent=2))\n\n\nif __name__==\'__main__\':main()\n', 'flow_operator/profiles.py': '"""Preserve the complete supplied spline, without a Fourier-family projection."""\nfrom dataclasses import dataclass\nfrom functools import lru_cache\nfrom pathlib import Path\n\nimport numpy as np\nfrom scipy.interpolate import CubicSpline\n\n\n@lru_cache(maxsize=16)\ndef quadrature(points):\n    z,w=np.polynomial.legendre.leggauss(points)\n    return (z+1)/2,w/2\n\n\n@dataclass\nclass Inlet:\n    y: np.ndarray\n    values: np.ndarray\n    mean_velocity: float\n    spline: CubicSpline\n    encoder_points: int = 257\n\n    @classmethod\n    def from_samples(cls, y, values, encoder_points=257):\n        y, values = np.asarray(y, float), np.asarray(values, float)\n        if y.ndim != 1 or values.shape != y.shape or len(y) < 4:\n            raise ValueError(\'Supply at least four (y, axial velocity) samples\')\n        if not np.isfinite(y).all() or not np.isfinite(values).all() or np.any(np.diff(y) <= 0):\n            raise ValueError(\'Coordinates must increase strictly; all samples must be finite\')\n        if abs(y[0]) > 1e-12 or abs(y[-1]-1) > 1e-12:\n            raise ValueError(\'The nondimensional inlet coordinate must span [0,1]\')\n        if np.max(abs(values[[0,-1]])) > 1e-9*max(1.,np.max(abs(values))):\n            raise ValueError(\'The no-slip walls require zero velocity at y=0 and y=1\')\n        if encoder_points < 16:\n            raise ValueError(\'Use at least 16 encoder quadrature points\')\n        spline = CubicSpline(y, values)\n        mean = float(spline.integrate(0,1))\n        if not mean > 1e-12:\n            raise ValueError(\'A positive mean is required by the chosen inlet-flow scaling\')\n        values = values/mean\n        return cls(y.copy(), values, mean, CubicSpline(y,values), encoder_points)\n\n    @classmethod\n    def from_csv(cls, path, encoder_points=257):\n        data = np.loadtxt(Path(path), delimiter=\',\',skiprows=1)\n        if data.ndim != 2 or data.shape[1] != 2:\n            raise ValueError(\'CSV must have header y,u followed by two numeric columns\')\n        return cls.from_samples(data[:,0],data[:,1],encoder_points)\n\n    def arrays(self, dtype=np.float32):\n        # Use a local t in [0,1] to avoid large inverse-mesh-power coefficients.\n        h = np.diff(self.y)\n        coeff = self.spline.c*np.stack([h**3,h**2,h,np.ones_like(h)])\n        integral = self.spline.antiderivative()\n        ey,ew = quadrature(self.encoder_points)\n        dense = np.linspace(0,1,max(1025,4*len(self.y)))\n        delta = self.spline(dense)-6*dense*(1-dense)\n        derivative = self.spline(dense,1)-(6-12*dense)\n        energy = np.trapezoid(delta**2,dense)\n        effective_mode = np.sqrt(np.trapezoid(derivative**2,dense)/max(energy,1e-12))/np.pi if energy>1e-10 else 1.\n        return {k:np.asarray(v,dtype=dtype) for k,v in dict(\n            knots=self.y, coeff=coeff, prefix=integral(self.y)-integral(0),\n            encoder_y=ey,encoder_w=ew,encoder_f=self.spline(ey),\n            effective_mode=np.clip(effective_mode,1,128)).items()}\n\n    def diagnostics(self):\n        y = np.linspace(0,1,max(4097,8*len(self.y)))\n        return dict(input_samples=len(self.y), encoder_points=self.encoder_points,\n            normalization_mean=self.mean_velocity, normalized_flux=float(self.spline.integrate(0,1)),\n            normalized_min=float(self.spline(y).min()), normalized_max=float(self.spline(y).max()),\n            local_reverse_flow=bool(self.spline(y).min() < -1e-8),\n            sample_preservation_max_error=float(np.max(abs(self.spline(self.y)-self.values))),\n            representation=\'C2 interpolating cubic spline through every supplied sample; no mode projection\',\n            resolution_notice=\'Samples alone cannot prove that an unknown underlying function is resolved.\')\n\n\ndef stack(profiles, dtype=np.float32):\n    values=[p.arrays(dtype) for p in profiles]\n    return {k:np.stack([p[k] for p in values]) for k in values[0]}\n', 'flow_operator/sampling.py': '"""Input functions only. No PDE responses, reference solver, or old imports."""\nfrom functools import lru_cache\n\nimport numpy as np\nfrom scipy.interpolate import CubicSpline\nfrom scipy.linalg import cholesky\nfrom .profiles import Inlet, stack\n\nFAMILIES=(\'grf\',\'oscillatory\',\'beta\',\'flat\',\'jets\',\'random_spline\',\'parabolic\')\n\n\n@lru_cache(maxsize=6)\ndef grf_factor(scale):\n    y=np.linspace(0,1,257)\n    return cholesky(np.exp(-.5*((y[:,None]-y[None,:])/scale)**2)+1e-10*np.eye(len(y)),lower=True)\n\n\ndef draw(rng,family,points=257):\n    y=np.linspace(0,1,points);base=6*y*(1-y)\n    if family==\'grf\':\n        g=grf_factor(float(rng.choice([.015,.03,.06,.12,.25,.4])))@rng.normal(size=257)\n        g=np.interp(y,np.linspace(0,1,257),g)\n        u=(y*(1-y))**rng.choice([1,2,4])*np.logaddexp(0,rng.uniform(.3,1.3)*g)\n    elif family==\'oscillatory\':\n        cycles=rng.integers(1,21,size=rng.integers(1,5))\n        a=rng.normal(size=len(cycles));a*=rng.uniform(.05,.75)/np.abs(a).sum()\n        u=base*(1+np.sum(a[:,None]*np.cos(2*np.pi*cycles[:,None]*y+rng.uniform(0,2*np.pi,len(a))[:,None]),axis=0))\n    elif family==\'beta\':\n        a,b=rng.integers(1,7,size=2)\n        if (a,b)==(4,4):a=5  # Reserve Beta(5,5) as an unseen profile.\n        u=y**a*(1-y)**b\n    elif family==\'flat\':\n        d=rng.uniform(.025,.2);power=rng.choice([1,2])\n        u=-np.expm1(-(y/d)**power)*-np.expm1(-((1-y)/d)**power)\n    elif family==\'jets\':\n        centers=rng.uniform(.1,.9,rng.integers(1,5));width=rng.uniform(.07,.25,len(centers))\n        u=np.sum(rng.uniform(.2,1.2,len(centers))[:,None]*np.exp(-((y-centers[:,None])/width[:,None])**2),axis=0)\n        u*=(-np.expm1(-y/.04))*(-np.expm1(-(1-y)/.04))\n    elif family==\'random_spline\':\n        knots=np.linspace(0,1,rng.integers(5,18))\n        g=CubicSpline(knots,rng.normal(size=len(knots)))(y)\n        u=(y*(1-y))**rng.choice([1,2,3])*np.logaddexp(0,g)\n    elif family==\'parabolic\':\n        u=base\n    else:raise ValueError(f\'Unknown family {family}\')\n    u[[0,-1]]=0\n    return Inlet.from_samples(y,u)\n\n\ndef make_pool(count=10000,seed=9703107,points=257,encoder_points=257):\n    rng=np.random.default_rng(seed);profiles=[];families=[]\n    # Retain the no-slip/unit-mean convention, with no four-mode restriction.\n    while len(profiles)<count:\n        # Include exact Poiseuille as an input condition, never as a response label.\n        family=str(rng.choice(FAMILIES,p=[.22,.25,.13,.12,.12,.12,.04]))\n        p=draw(rng,family,points)\n        check=p.spline(np.linspace(0,1,1025))\n        if check.min() < -1e-7 or check.max()>3.0:continue\n        p.encoder_points=encoder_points;profiles.append(p);families.append(family)\n    data=stack(profiles)\n    return data,dict(seed=seed,count=count,input_points=points,encoder_points=encoder_points,\n        families={f:families.count(f) for f in FAMILIES},response_labels_present=False,\n        reserved_profiles=[\'exact Beta(5,5); other diagnostic profiles are checked against the assembled pool\'],\n        training_speed_cap=3.,profile_source=\'seven input-only families; no CFD\',\n        note=\'Training coverage is finite; the inference interface does not restrict profiles to these generators.\')\n\n\ndef diagnostic_profiles(points=257,encoder_points=257):\n    y=np.linspace(0,1,points)\n    raw=[(\'poiseuille\',6*y*(1-y),40.),\n         (\'nine_oscillations\',6*y*(1-y)*(1-.6*np.cos(18*np.pi*y)),55.),\n         (\'beta_5_5\',630*y**4*(1-y)**4,30.),\n         (\'flat\',(-np.expm1(-(y/.05)**2))*(-np.expm1(-((1-y)/.05)**2)),70.)]\n    profiles=[Inlet.from_samples(y,u,encoder_points) for _,u,_ in raw]\n    return stack(profiles),np.array([r for _,_,r in raw],np.float32),[name for name,_,_ in raw]\n', 'flow_operator/train.py': '"""Physics-only training: inputs and NS/traction residuals, no CFD labels.\n\nReference metrics may be read by a separate monitor, never by loss/batch/scale.\n"""\nimport os\nos.environ.setdefault(\'XLA_PYTHON_CLIENT_PREALLOCATE\',\'false\')\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\nimport time\n\nimport jax\nimport jax.numpy as jnp\nimport numpy as np\nfrom .model import Config,initialize,make_functions\nfrom .sampling import make_pool,diagnostic_profiles\nfrom .checkpoint import save,load\n\n\ndef make_step(functions,normalization=\'none\'):\n    def loss(params,batch):\n        if normalization in [\'input_curvature\',\'input_curvature_sqrt\']:\n            # Exact integral of (f\'\')^2 for each supplied cubic. This uses ONLY\n            # input derivatives and Re, never response fields or CFD statistics.\n            profile=batch[\'profiles\'];a,b=profile[\'coeff\'][:,0],profile[\'coeff\'][:,1]\n            h=jnp.diff(profile[\'knots\'],axis=1)\n            curvature=jnp.sqrt(jnp.maximum(jnp.sum((12*a*a+12*a*b+4*b*b)/h**3,axis=1),0))\n            scale=1+curvature/batch[\'reynolds\']\n            r=functions[\'residual\'](params,profile,batch[\'reynolds\'],batch[\'xy\'])\n            o=functions[\'outlet\'](params,profile,batch[\'reynolds\'],batch[\'outlet_y\'])\n            exponent=2 if normalization==\'input_curvature\' else 1\n            terms=jnp.array([jnp.mean(jnp.sum(r[...,:2]**2,axis=-1)/scale[:,None]**exponent),\n                             jnp.mean(jnp.sum(o**2,axis=-1)),jnp.mean(r[...,2]**2)])\n        else:\n            terms=functions[\'raw_terms\'](params,batch)\n        return terms[0]+10*terms[1],terms\n    @jax.jit\n    def step(params,m,v,count,batch):\n        (loss_value,terms),grad=jax.value_and_grad(loss,has_aux=True)(params,batch)\n        norm=jnp.sqrt(sum(jnp.sum(g*g) for g in jax.tree_util.tree_leaves(grad)))\n        grad=jax.tree_util.tree_map(lambda g:g*jnp.minimum(1.,100/(norm+1e-12)),grad)\n        m=jax.tree_util.tree_map(lambda a,g:.9*a+.1*g,m,grad)\n        v=jax.tree_util.tree_map(lambda a,g:.999*a+.001*g*g,v,grad)\n        lr=jnp.maximum(2e-5,1e-3*.95**(count/2000))\n        params=jax.tree_util.tree_map(lambda p,a,b:p-lr*(a/(1-.9**count))/(jnp.sqrt(b/(1-.999**count))+1e-8),params,m,v)\n        return params,m,v,loss_value,terms,norm\n    return step\n\n\ndef batch(rng,pool,batch_size,points,length=5.):\n    indices=rng.integers(len(pool[\'knots\']),size=batch_size)\n    profiles={k:v[indices] for k,v in pool.items()}\n    re=np.exp(rng.uniform(np.log(10),np.log(100),batch_size)).astype(np.float32)\n    # Half of conditions explicitly cover both ends of the Re range.\n    edge=rng.random(batch_size)<.2;re[edge]=rng.choice([10.,100.],edge.sum())\n    xy=rng.uniform(size=(batch_size,points,2)).astype(np.float32)\n    # Cover the full domain and the short development lengths of rough inlets.\n    choose=rng.random(xy.shape[:-1])<.65\n    xy[...,0]=length*np.where(choose,xy[...,0]**3,xy[...,0])\n    return dict(profiles=profiles,reynolds=re,xy=xy,\n                outlet_y=rng.uniform(size=(batch_size,48)).astype(np.float32))\n\n\ndef train(args):\n    output=Path(args.output);output.mkdir(parents=True,exist_ok=True)\n    if args.require_gpu and not any(d.platform==\'gpu\' for d in jax.devices()):raise RuntimeError(\'GPU required\')\n    config=Config(width=args.width,latent=args.latent,depth=args.depth,lift=args.lift,functional_moments=args.functional_moments,\n                  encoder_kind=args.encoder_kind,sensor_count=args.sensor_count)\n    rng=np.random.default_rng(91838);params=initialize(config);first=0\n    m,v=jax.tree_util.tree_map(jnp.zeros_like,params),jax.tree_util.tree_map(jnp.zeros_like,params)\n    if args.resume:\n        params,old_config,metadata,state=load(args.resume,optimizer=True)\n        if old_config!=config:raise ValueError(\'Resume configuration differs from CLI configuration\')\n        if metadata[\'pool_seed\']!=args.pool_seed or metadata[\'pool_count\']!=args.pool_count:raise ValueError(\'Resume pool changed\')\n        first=metadata[\'step\'];rng.bit_generator.state=metadata[\'rng_state\'];m,v=state\n    if args.steps<=first:raise ValueError(\'Target steps must exceed current checkpoint step\')\n    pool,pool_metadata=make_pool(args.pool_count,args.pool_seed,encoder_points=args.encoder_points)\n    (output/\'input_pool_metadata.json\').write_text(json.dumps(pool_metadata,indent=2))\n    pool_digest=hashlib.sha256(b\'\'.join(v.tobytes() for v in pool.values())).hexdigest()\n    if args.resume and metadata[\'pool_sha256\']!=pool_digest:raise ValueError(\'Resume input pool or its representation changed\')\n    functions=make_functions(config);update=make_step(functions,args.physics_normalization)\n    reference_cases=None\n    if args.reference_monitor:\n        from .validation import read_cases\n        reference_cases=read_cases(args.reference_monitor,args.encoder_points)\n    held,re,names=diagnostic_profiles(encoder_points=args.encoder_points)\n    for i,name in enumerate(names):\n        if name!=\'poiseuille\' and np.any(np.all(pool[\'coeff\']==held[\'coeff\'][i],axis=(1,2))):\n            raise ValueError(f\'Exact diagnostic profile leaked into the input training bank: {name}\')\n    vrng=np.random.default_rng(918390)\n    # Fixed and independent of the optimizer RNG, with entrance coverage.\n    points=vrng.uniform(size=(len(re),128,2)).astype(np.float32)\n    points[...,0]=config.length*points[...,0]**2\n    yy=np.broadcast_to(np.linspace(.001,.999,96,dtype=np.float32),(len(re),96)).copy()\n    start=time.perf_counter()\n    def monitor(step):\n        residual=np.asarray(functions[\'residual\'](params,held,re,points))\n        boundary=np.asarray(functions[\'outlet\'](params,held,re,yy))\n        records=[dict(name=name,reynolds=float(r),momentum_rms=float(np.sqrt(np.mean(np.sum(rr[:,:2]**2,axis=-1)))),\n                      divergence_rms=float(np.sqrt(np.mean(rr[:,2]**2))),\n                      outlet_traction_rms=float(np.sqrt(np.mean(np.sum(oo**2,axis=-1)))))\n                 for name,r,rr,oo in zip(names,re,residual,boundary)]\n        return dict(step=step,seconds=time.perf_counter()-start,role=\'repeated_physics_validation_no_CFD_labels\',cases=records)\n    with (output/\'training.jsonl\').open(\'a\',buffering=1) as log:\n        initial=monitor(first);print(json.dumps(initial),flush=True);log.write(json.dumps(initial)+\'\\n\')\n        for count in range(first+1,args.steps+1):\n            b=batch(rng,pool,args.batch,args.points)\n            params,m,v,value,terms,gradient_norm=update(params,m,v,jnp.array(count),b)\n            if count%args.log_every==0 or count==args.steps or count==first+1:\n                record=monitor(count);record.update(loss=float(value),momentum_mse=float(terms[0]),\n                    outlet_mse=float(terms[1]),divergence_mse=float(terms[2]),gradient_l2=float(gradient_norm),\n                    physics_normalization=args.physics_normalization,\n                    momentum_mse_is_normalized=args.physics_normalization!=\'none\')\n                if not np.isfinite(record[\'loss\']):raise RuntimeError(\'Non-finite objective\')\n                if reference_cases and (count%args.reference_every==0 or count==args.steps):\n                    from .validation import compare\n                    record[\'reference_validation\']=compare(params,functions,reference_cases)\n                print(json.dumps(record),flush=True);log.write(json.dumps(record)+\'\\n\')\n                save(output/\'checkpoint.npz\',params,config,count,rng.bit_generator.state,(m,v),\n                    dict(pool_seed=args.pool_seed,pool_count=args.pool_count,pool_sha256=pool_digest,\n                         encoder_points=args.encoder_points,training_mode=\'physics_only\',CFD_response_labels_used=0,\n                         physics_normalization=args.physics_normalization,\n                         validation_status=\'not_yet_independently_accepted\',diagnostics=record))\n    (output/\'summary.json\').write_text(json.dumps(dict(step=count,seconds=time.perf_counter()-start,\n        devices=[str(d) for d in jax.devices()],config=config.json(),last=record,training_mode=\'physics_only\',\n        CFD_response_labels_used=0,validated_for_deployment=False),indent=2))\n\n\ndef main():\n    p=argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--output\',required=True);p.add_argument(\'--steps\',type=int,default=50000)\n    p.add_argument(\'--batch\',type=int,default=8);p.add_argument(\'--points\',type=int,default=64)\n    p.add_argument(\'--pool-count\',type=int,default=10000);p.add_argument(\'--pool-seed\',type=int,default=9703107)\n    p.add_argument(\'--encoder-points\',type=int,default=257);p.add_argument(\'--width\',type=int,default=96)\n    p.add_argument(\'--latent\',type=int,default=48);p.add_argument(\'--depth\',type=int,default=3)\n    p.add_argument(\'--log-every\',type=int,default=250);p.add_argument(\'--resume\');p.add_argument(\'--require-gpu\',action=\'store_true\')\n    p.add_argument(\'--reference-monitor\');p.add_argument(\'--reference-every\',type=int,default=1000)\n    p.add_argument(\'--lift\',choices=[\'gaussian\',\'oseen\'],default=\'gaussian\')\n    p.add_argument(\'--physics-normalization\',choices=[\'none\',\'input_curvature\',\'input_curvature_sqrt\'],default=\'none\')\n    p.add_argument(\'--functional-moments\',action=\'store_true\')\n    p.add_argument(\'--encoder-kind\',choices=[\'pooled\',\'sensor\'],default=\'pooled\')\n    p.add_argument(\'--sensor-count\',type=int,default=257)\n    a=p.parse_args()\n    if min(a.steps,a.batch,a.points,a.pool_count,a.log_every,a.reference_every,a.width,a.latent,a.depth)<1:p.error(\'Counts and network sizes must be positive\')\n    train(a)\n\n\nif __name__==\'__main__\':main()\n', 'flow_operator/validation.py': '"""Read-only reference comparison. Never imported into the optimizer objective."""\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport jax\nimport numpy as np\nfrom .profiles import Inlet,stack\n\n\ndef read_cases(directory,encoder_points=257):\n    cases=[]\n    for path in sorted(Path(directory).glob(\'*.npz\')):\n        with np.load(path,allow_pickle=False) as d:\n            inlet=Inlet.from_samples(d[\'inlet_y\'],d[\'inlet_values\'],encoder_points)\n            cases.append(dict(name=path.stem,profile=inlet,re=float(d[\'reynolds\']),xy=d[\'xy\'].copy(),\n                              reference=d[\'reference\'].copy(),metadata=json.loads(str(d[\'metadata\']))))\n    if not cases:raise ValueError(\'No reference cases found\')\n    return cases\n\n\ndef compare(params,functions,cases):\n    records=[]\n    dtype=np.float64 if jax.config.x64_enabled else np.float32\n    for c in cases:\n        profile=stack([c[\'profile\']],dtype);predictions=[]\n        for xy in np.array_split(c[\'xy\'],int(np.ceil(len(c[\'xy\'])/512))):\n            predictions.append(np.asarray(functions[\'fields\'](params,profile,np.array([c[\'re\']],dtype),xy[None].astype(dtype)))[0])\n        pred=np.concatenate(predictions);truth=c[\'reference\'];error=pred-truth\n        velocity=float(np.linalg.norm(error[:,:2])/np.linalg.norm(truth[:,:2]))\n        pressure=float(np.linalg.norm(error[:,2])/max(np.linalg.norm(truth[:,2]),1e-12))\n        records.append(dict(name=c[\'name\'],reynolds=c[\'re\'],velocity_relative_l2=velocity,\n            u_relative_l2=float(np.linalg.norm(error[:,0])/np.linalg.norm(truth[:,0])),\n            v_relative_l2=float(np.linalg.norm(error[:,1])/max(np.linalg.norm(truth[:,1]),1e-12)),\n            v_rmse=float(np.sqrt(np.mean(error[:,1]**2))),pressure_relative_l2=pressure,\n            role=\'repeated_reference_validation_excluded_from_objective\'))\n    return dict(role=\'read_only_validation_not_backpropagated\',cases=records,\n                max_velocity_relative_l2=max(r[\'velocity_relative_l2\'] for r in records),\n                max_pressure_relative_l2=max(r[\'pressure_relative_l2\'] for r in records))\n', 'tests/conftest.py': "import os\nos.environ.setdefault('XLA_PYTHON_CLIENT_PREALLOCATE','false')\nimport jax\njax.config.update('jax_enable_x64',True)\n", 'tests/test_operator.py': "import json\nfrom types import SimpleNamespace\n\nimport jax\nimport jax.numpy as jnp\nimport numpy as np\nimport pytest\nfrom scipy.signal import find_peaks\n\nfrom flow_operator.profiles import Inlet,stack\nfrom flow_operator.model import Config,initialize,make_functions,spline,context,point_fields\nfrom flow_operator.checkpoint import save,load\nfrom flow_operator.api import FlowOperator\n\n\n@pytest.fixture(scope='module',params=['gaussian','oseen','oseen_moments','oseen_sensor'])\ndef setup_model(request):\n    c=Config(width=12,latent=6,depth=2,encoder_width=12,lift='oseen' if request.param.startswith('oseen') else 'gaussian',\n             functional_moments=request.param=='oseen_moments',encoder_kind='sensor' if request.param=='oseen_sensor' else 'pooled')\n    p=jax.tree_util.tree_map(lambda z:z.astype(jnp.float64),initialize(c))\n    return c,p,make_functions(c)\n\n\n@pytest.mark.parametrize('kind',['nine','twenty_five','beta','flat','asymmetric'])\ndef test_full_input_is_preserved_at_boundary(kind,setup_model):\n    c,p,f=setup_model;y=np.linspace(0,1,257)\n    values={'nine':6*y*(1-y)*(1-.6*np.cos(18*np.pi*y)),\n            'twenty_five':6*y*(1-y)*(1-.6*np.cos(50*np.pi*y)),\n            'beta':630*y**4*(1-y)**4,\n            'flat':(-np.expm1(-(y/.05)**2))*(-np.expm1(-((1-y)/.05)**2)),\n            'asymmetric':y*(1-y)*np.exp(-((y-.3)/.15)**2)}[kind]\n    inlet=Inlet.from_samples(y,values,65);data=stack([inlet],np.float64)\n    dense=np.linspace(0,1,1025)\n    xy=np.column_stack([np.zeros_like(dense),dense])[None]\n    # Nonzero learned correction, and then a large parameter perturbation.\n    for params in [p,jax.tree_util.tree_map(lambda a:a+.03,p)]:\n        out=np.asarray(f['fields'](params,data,np.array([55.]),xy))[0]\n        np.testing.assert_allclose(out[:,0],inlet.spline(dense),atol=1e-10)\n        assert np.max(abs(out[:,1]))<1e-10\n        walls=np.array([[[x,w] for w in [0.,1.] for x in [.01,1.,4.99]]])\n        assert np.max(abs(np.asarray(f['fields'](params,data,np.array([55.]),walls))[...,:2]))<1e-9\n    if kind=='nine':assert len(find_peaks(out[:,0])[0])==9\n    if kind=='twenty_five':assert len(find_peaks(out[:,0])[0])==25\n\n\ndef test_spline_integral_and_spatial_derivatives(setup_model):\n    y=np.sort(np.r_[0.,np.random.default_rng(9).uniform(size=40),1.])\n    inlet=Inlet.from_samples(y,6*y*(1-y)*(1+.2*np.cos(18*np.pi*y)),65)\n    a={k:jnp.array(v) for k,v in inlet.arrays(np.float64).items()}\n    for z in [.1234,.4321,.8765]:\n        f,I=spline(a,z)\n        np.testing.assert_allclose(f,inlet.spline(z),atol=1e-10)\n        np.testing.assert_allclose(I,inlet.spline.antiderivative()(z)-inlet.spline.antiderivative()(0),atol=1e-10)\n        np.testing.assert_allclose(jax.grad(lambda t:spline(a,t)[1])(z),f,atol=1e-9)\n        np.testing.assert_allclose(jax.grad(jax.grad(lambda t:spline(a,t)[0]))(z),inlet.spline(z,2),atol=1e-7)\n\n\ndef test_exact_poiseuille_and_outlet_and_mass(setup_model):\n    c,p,f=setup_model\n    zero=jax.tree_util.tree_map(jnp.zeros_like,p)\n    y=np.linspace(0,1,65);inlet=Inlet.from_samples(y,6*y*(1-y),65)\n    data=stack([inlet],np.float64)\n    xy=np.array([[[.123,.234],[1.73,.712],[4.81,.323]]])\n    out=np.asarray(f['fields'](zero,data,np.array([40.]),xy))[0]\n    truth=np.column_stack([6*xy[0,:,1]*(1-xy[0,:,1]),np.zeros(3),12/40*(5-xy[0,:,0])])\n    np.testing.assert_allclose(out,truth,atol=1e-9)\n    assert np.max(abs(np.asarray(f['residual'](zero,data,np.array([40.]),xy))))<1e-8\n    assert np.max(abs(np.asarray(f['outlet'](zero,data,np.array([40.]),np.array([[.13,.37,.81]])))))<1e-9\n    r=np.asarray(f['residual'](p,data,np.array([40.]),xy))\n    assert np.max(abs(r[...,2]))<1e-10\n\n\ndef test_residual_against_independent_finite_differences(setup_model):\n    c,p,f=setup_model;y=np.linspace(0,1,257)\n    inlet=Inlet.from_samples(y,6*y*(1-y)*(1+.3*np.cos(18*np.pi*y)),65)\n    a={k:jnp.array(v) for k,v in inlet.arrays(np.float64).items()}\n    re=55.;b=context(p,a,re,c);z=np.array([.071,.3213]);h=1e-5\n    fn=lambda q:np.asarray(point_fields(p,b,a,re,jnp.array(q),c))\n    value=fn(z);u,v,_=value\n    jac=np.column_stack([(fn(z+np.eye(2)[k]*h)-fn(z-np.eye(2)[k]*h))/(2*h) for k in range(2)])\n    lap=sum((fn(z+np.eye(2)[k]*h)-2*value+fn(z-np.eye(2)[k]*h))/h**2 for k in range(2))\n    finite=np.array([u*jac[0,0]+v*jac[0,1]+jac[2,0]-lap[0]/re,\n                     u*jac[1,0]+v*jac[1,1]+jac[2,1]-lap[1]/re,jac[0,0]+jac[1,1]])\n    actual=np.asarray(f['residual'](p,stack([inlet],np.float64),np.array([re]),z[None,None]))[0,0]\n    np.testing.assert_allclose(actual,finite,rtol=2e-4,atol=2e-4)\n\n\ndef test_encoder_accepts_different_quadrature_and_input_sizes(setup_model):\n    c,p,f=setup_model\n    for n,q in [(65,33),(257,129),(513,257)]:\n        y=np.linspace(0,1,n);inlet=Inlet.from_samples(y,630*y**4*(1-y)**4,q)\n        out=f['fields'](p,stack([inlet],np.float64),np.array([55.]),np.array([[[.31,.43]]]))\n        assert np.isfinite(np.asarray(out)).all()\n\n\ndef test_checkpoint_roundtrip_and_resume_state(tmp_path,setup_model):\n    c,p,f=setup_model\n    rng=np.random.default_rng(31);rng.uniform(size=17)\n    path=tmp_path/'checkpoint.npz'\n    save(path,p,c,7,rng.bit_generator.state,(p,p),dict(validation_status='not_accepted'))\n    got,config,meta,state=load(path,optimizer=True)\n    assert config==c and meta['step']==7\n    for a,b in zip(jax.tree_util.tree_leaves(p),jax.tree_util.tree_leaves(got)):np.testing.assert_array_equal(a,b)\n    other=np.random.default_rng();other.bit_generator.state=meta['rng_state']\n    np.testing.assert_array_equal(other.uniform(size=10),rng.uniform(size=10))\n\n\ndef test_reusable_inference_requires_explicit_provisional_status(tmp_path,setup_model):\n    c,p,f=setup_model;path=tmp_path/'candidate.npz'\n    save(path,p,c,7,np.random.default_rng(3).bit_generator.state)\n    with pytest.raises(ValueError,match='passing benchmark'):FlowOperator(path)\n    op=FlowOperator(path,allow_unvalidated=True,chunk_size=32)\n    y=np.linspace(0,1,257);profile=Inlet.from_samples(y,6*y*(1-y)*(1-.6*np.cos(18*np.pi*y)),33)\n    coords=np.column_stack([np.zeros(33),np.linspace(0,1,33)])\n    first=op.evaluate(profile,55.,coords);second=op.evaluate(profile,55.,coords)\n    np.testing.assert_array_equal(first,second)\n    np.testing.assert_allclose(first[:,0],profile.spline(coords[:,1]),atol=2e-6)\n", 'pyproject.toml': '[tool.pytest.ini_options]\ntestpaths = ["tests"]\nnorecursedirs = ["old_verssion", "runs", "artifacts", ".git"]\n', 'configs/acceptance.json': '{\n  "description": "Fixed before reviewing new CFD validation results; benchmark-specific, not a universal error guarantee.",\n  "velocity_relative_l2": 0.05,\n  "pressure_relative_l2": 0.10,\n  "momentum_rms": 0.05,\n  "divergence_rms": 0.0001,\n  "outlet_traction_rms": 0.01,\n  "flux_max_error": 0.002,\n  "boundary_max_error": 0.0001\n}\n'}
for name,text in sources.items():
    path=Path(name)
    path.parent.mkdir(parents=True,exist_ok=True)
    path.write_text(text)
reference_files={'validation/references/asymmetric.npz': '...', 'validation/references/flat.npz': '...', 'validation/references/nine.npz': '...', 'validation/references/beta.npz': '...'}
for name,text in reference_files.items():
    path=Path(name)
    path.parent.mkdir(parents=True,exist_ok=True)
    path.write_bytes(base64.b64decode(text))
Path("output").mkdir(exist_ok=True)
Path("output/source_manifest.json").write_text('{\n  "flow_operator/__init__.py": "948b04abf5bc637cf30ed17c473f3b70c09ae25ff1bc39f432019bf65870ed69",\n  "flow_operator/api.py": "637cbe14999ae207e59cfff9f3f714fbcb6de2a1edb3a043e013624f26451072",\n  "flow_operator/checkpoint.py": "aec584b74529bbdfe724f7a50e07ce055d1047a819661cd533327e36d938e40f",\n  "flow_operator/evaluate.py": "eacecbf9af4cbd34e6cc95b118551e784d0ae814782c443d097aec5f29b89073",\n  "flow_operator/model.py": "3eaca8e5cb5a635f9b8863fbf8e451051a9f9f949580138825892213b1fff99a",\n  "flow_operator/predict.py": "27767cc7c3bba817203dfcbf5c2d02f2c2648590dd0fdfab3727f625f4585c55",\n  "flow_operator/profiles.py": "6088de44aab003fc6148afb8bf43b35ef758c817ae585c139d91bc344f8658cd",\n  "flow_operator/sampling.py": "fd9fcd296f3a7359915c973debee72335486e06a9e991583e8cd6582a37e6d4b",\n  "flow_operator/train.py": "d52f276605a374244e7dfe05e5e982f4d7e011a802377a4170621baddc26089c",\n  "flow_operator/validation.py": "2061225b31b86980bbf1f315b108e5de0a9b578b78cdbd84c1b43feb32664461",\n  "tests/conftest.py": "1b86364544c418af2e1d15acfe13efe225763e969ff3f232d5e87bb9176cb580",\n  "tests/test_operator.py": "8d1504222a147b464961fe87aa77e8d31b188588cb5e87eb2c74602645761da7",\n  "pyproject.toml": "62f4cd5c826d3092298b96db115e7a807de613ebc2980a991c3752d643a3906b",\n  "configs/acceptance.json": "ad2b207f8202c2e642e436bcde457f2d42cc1ee33917e15a5d479817f5d84c25"\n}')
subprocess.run([sys.executable,"-m","pytest","-q"],check=True)


In [ ]:
command=['-m', 'flow_operator.train', '--output', 'output', '--steps', '5000', '--batch', '64', '--points', '48', '--pool-count', '10000', '--log-every', '250', '--require-gpu', '--width', '128', '--latent', '64', '--depth', '4', '--lift', 'oseen', '--physics-normalization', 'input_curvature_sqrt', '--encoder-kind', 'sensor', '--reference-monitor', 'validation/references', '--reference-every', '1000']
subprocess.run([sys.executable,*command],check=True)


In [ ]:
print(Path("output/summary.json").read_text())
